# Hierarchical Clustering

In [1]:
# Import packages
import hicona as hi

# Data paths 
TABLE_PATH = "../data/IMR90_chr17_annotated_table"

Clustering is a focal point in HiCONA. Most of the graph analysis provided by HiCONA has clustering as a starting point, whether it is uncertain network reconstruction, edge/vertex marginal probability, edge shuffling or clustering per se. Clustering at its core is performed using functions provided by `graph-tool`, with suitable parameters for Hi-C like data and the due modifications.

## Starting point

As you might expect, one of the possibile starting points for network clustering is an instance of `HiconaGraph`; simply generate a graph, and call the `compute_clustering()` method with the desired set of parameters (see following section). After clustering, it is likely that you will want to then convert the `HiconaGraph` back into a `PixelTable` for further analysis and/or plotting. For this reason, HiCONA also allows to perform clustering starting from a `PixelTable` using the `add_clustering()` method; under the hood, HiCONA generates the graph, computes clustering and adds the results back in the original `PixelTable` as new columns. This allows to simplify a somewhat tricky and recurrent procedure at the expense of some flexibility. Clustering will be presented using the `PixelTable` API.

In [2]:
table = hi.PixelTable.load(TABLE_PATH)

## Clustering Modalities

`table.add_clustering` has 4 parameters:

- `region`: the genomic region to cluster. 
- `marginals`: which marginal probabilities to compute. This defines the model used for the clustering and by proxy running time.
- `seed`: rng seed for result reproducibility.
- `logging_level`: how much information do print to console. Level is specified using the Python `logging` convention.

<div class="alert alert-warning">

Warning

If a genomic region is not specified, the clustering will be computed using the entire binning present in the table. This is likely not what you would like and it could also make execution take forever. Always specify a region unless you have a very good reason not to do so.

</div>

The most relevant aspect of your clustering is the choice of the `marginals` parameters, so let us go through the options.

### Modality: `no`

The default choice for `marginals` is `no`. In this case, internally, `add_clustering` computes an initial rough clustering using description length minimization on a nested block model; the clustering is then refined using a variable temperature simulated annealing algorithm.

In [3]:
table.add_clustering("chr17:0-10000000", marginals="no")

INFO:Clustering:Started graph clustering (marginals=no, seed=42)
INFO:Clustering:Computing rough clustering (description length minimization)
INFO:Clustering:Refining clustering (simulated annealing)
INFO:Clustering:Projecting clustering on the graph


In [4]:
table.get_dataframe().head()  # Pixels

bin1_id,bin2_id,count,rand_int,level_(0),level_(1)
i64,i64,i32,i64,i32,i32
26475,26475,17,73,null,null
26475,26481,1,47,13,3
26475,26518,1,53,0,0
26475,26526,1,92,0,3
26475,26534,1,95,0,0


<div class="alert alert-info">

Note

Self looping pixels are assigned `null` since they are not inserted in the graph when computing clustering.

</div>

In [5]:
table.bins.get_dataframe("chr17:0-10000000").head()  # Bins

bin_id,chrom,start,end,weight,level_(0),level_(1)
i64,str,i32,i32,f64,i32,i32
26475,"""chr17""",0,100000,null,13,3
26476,"""chr17""",100000,200000,0.007338,13,3
26477,"""chr17""",200000,300000,0.006855,13,3
26478,"""chr17""",300000,400000,0.005692,13,3
26479,"""chr17""",400000,500000,0.007215,13,3


This procedure allows to compute a node(bin)-level hierarchical clustering of the graph. Multiple columns are added to the bin table, called `level_(0)`, ..., `level_(N)`. Level 0 is the lowest in the hierarchical structure (smaller clusters) while level `N` is the highest (bigger clusters).

Since the clustering is at node level, it needs to be projected on edge(pixel)-level. For each level of the hierarchy, a pixel is assigned to cluster `x` if both bins composing it belong to cluster `x` at that level; if that is not the case, the pixel is assigned to `0`, meaning no cluster. Rephrasing, intra-cluster pixels are assigned their cluster number, while inter-cluster pixels are assigned the value `0`.

This is the fastest clustering option and should complete in a reasonable amount of time even for full-chromosome sized graphs.

### Modality: `bins`

`bins` extends upon `no` by also adding the `bin_prob` column to the bin table. `bin_prob` corresponds to the probability with which the bin was assigned to its cluster at level 0. This value is computed by performing iterations of Markov-Chain Monte Carlo Equilibration of the clustering.

In [6]:
table = hi.PixelTable.load(TABLE_PATH)
table.add_clustering("chr17:0-10000000", marginals="bins")

INFO:Clustering:Started graph clustering (marginals=bins, seed=42)
INFO:Clustering:Computing rough clustering (description length minimization)
INFO:Clustering:Refining clustering (simulated annealing)
INFO:Clustering:Computing bin marginals (equilibration)
INFO:Clustering:Projecting clustering on the graph


In [7]:
table.bins.get_dataframe("chr17:0-10000000").head()  # Bins

bin_id,chrom,start,end,weight,bin_prob,level_(0),level_(1)
i64,str,i32,i32,f64,f64,i32,i32
26475,"""chr17""",0,100000,null,1.0,1,1
26476,"""chr17""",100000,200000,0.007338,1.0,1,1
26477,"""chr17""",200000,300000,0.006855,1.0,1,1
26478,"""chr17""",300000,400000,0.005692,1.0,1,1
26479,"""chr17""",400000,500000,0.007215,1.0,1,1


<div class="alert alert-info">

Note

This probability could hypothetically be computed for any level of the hierarchy but, in an effort to maintain results interpretable, it is computed only for level 0 since the higher level of the hierarchy depend on this level.

</div>

This option is a bit more expensive than `no` but should still have manageable runtimes.

### Modality: `pixels`

The biggest novelty provided by HiCONA is probably the ability to perform uncertain network reconstruction on a Hi-C data derived network. In short, we assume that the Hi-C graph was generated through repetive noisy measurements of a real network; we then try to model both the original network as well as the noisy measurments (allowing for heterogenous errors) and estimate the posterior probabilities of all edges using a MCMC simulation procedure. For an in-depth mathematical explanation of the procedure, please refer to the [original implementation](https://graph-tool.skewed.de/static/doc/demos/reconstruction_direct/reconstruction.html).

First of all, this procedure adds edges which are deemed important for the structure but are not in the original graph (the assumption is that they "should" be there according to the estimates but they are not due to noisy measurment). Moreover, each edge (whether it was reconstructed or not) is assigned a posterior probability of being "real" and not just a result of noise. This probability can be used to filter the graph to a desired level of confidence.

Aside from all the same columns from the previously described modalities (cluster levels and bin probabilities), two new columns are added to the pixel tables by this procedure:

- `pix_prob`, containing the posterior probabilites of the edges
- `pix_group`, which is a categorical property with value
  - `0` if the pixel was in the graph prior to the reconstruction and still is
  - `1` if the pixel was not in the original graph but it is in the reconstructed one
  - `2` if the pixel was in the original graph but it is not in the reconstructed one (this case is in general the result of not enough simulation time)
    

In [8]:
table = hi.PixelTable.load(TABLE_PATH)
table.add_clustering("chr17:0-10000000", marginals="pixels")

INFO:Clustering:Started graph clustering (marginals=pixels, seed=42)
INFO:Clustering:Computing rough clustering (description length minimization)
INFO:Clustering:Refining clustering (simulated annealing)
INFO:Clustering:Computing bin and pixels marginals (network reconstruction)
INFO:Clustering:Projecting clustering on the graph


In [9]:
table.get_dataframe().head()  # Pixels

bin1_id,bin2_id,count,rand_int,pix_prob,pix_group,level_(0),level_(1),level_(2),level_(3)
i64,i64,i32,i64,f64,i32,i32,i32,i32,i32
26475,26475,17,73,null,null,null,null,null,null
26475,26476,1,null,0.00454,0,0,0,2,2
26475,26477,0,null,0.00802,1,0,0,2,2
26475,26478,0,null,0.0066,1,0,0,2,2
26475,26479,0,null,0.0046,1,0,0,2,2


In [10]:
table.bins.get_dataframe("chr17:0-10000000").head()  # Bins

bin_id,chrom,start,end,weight,bin_prob,level_(0),level_(1),level_(2),level_(3)
i64,str,i32,i32,f64,f64,i32,i32,i32,i32
26475,"""chr17""",0,100000,null,0.162323,10,4,2,2
26476,"""chr17""",100000,200000,0.007338,0.99484,1,1,2,2
26477,"""chr17""",200000,300000,0.006855,0.99482,1,1,2,2
26478,"""chr17""",300000,400000,0.005692,0.99484,1,1,2,2
26479,"""chr17""",400000,500000,0.007215,0.99484,1,1,2,2


<div class="alert alert-info">

Note

Due to the reconstruction step, the length of the PixelTable might change.

</div>

Although this modality provides a lot of information, it can also be quite slow, as it is MCMC based and not easily parallelizable.

## Clustering Tips

- Clustering is affected by region boundaries position: if you subset a chromosome to an arbitrary genomic region of interest, you are inevitably removing some context, which will make clustering less accurate, especially at the borders of the region. When possible, perform clustering on the entire chromosome and then look at your region of interest. If this is not possible (usually due to very small bin size), consider widening the area by some amount and subsetting after clustering.
- While running clustering can be quite slow, especially with modality `pixels`, the algorithm is very efficient in terms of memory (usually the required memory is only slightly above `O(n)`). For this reason, if you are interested in multiple regions, the proper way to speed up computation is to run multiple clustering in parallel with a couple cores each, rather than running one clustering at a time with loads of resources (as a very small amount of them will be used anyway).
- Always save your `PixelTable` after clustering. Saved tables do not take much disk space and allow you to re-analyze and plot your data later on without having to re-run the clustering.